In [4]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import base64 

load_dotenv()

True

In [5]:
with open("blood_report.png","rb") as f:
    image_b64 =base64.b64encode(f.read()).decode()

image_b64[:200]

'iVBORw0KGgoAAAANSUhEUgAAAjwAAAK5CAYAAAC7cvTxAAAAAXNSR0IArs4c6QAAAARnQU1BAACxjwv8YQUAAAAJcEhZcwAAEnQAABJ0Ad5mH3gAAKWwSURBVHhe7d0xi/Nc2Cf2/5OFhYV0+5YLGhc3dqGFrdJq3dgpJhAik1qkGC8sUw0EgZttDGZhIDAp1k6jD2CR'

In [9]:
llm= ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

message=HumanMessage(content=[
      {"type":"image_url","image_url":{"url":f"data:image/png;base64,{image_b64}"}},
      {"type":"text","text":"this is a blood report,extract all test results and flag any values outside normal range as abnormal"}
    ])

response=llm.invoke([message])
print(response.content)

Here are the test results extracted from the blood report with values outside the normal range flagged as abnormal:

**COMPLETE BLOOD COUNT (CBC)**

1. Hemoglobin: 15.1 g/dL (Normal: 13.5-17.5) - **Normal**
2. Hematocrit: 44% (Normal: 41-53%) - **Normal**
3. WBC: 6.8 x10^3/uL (Normal: 4.5-11.0) - **Normal**
4. Platelets: 220 x10^3/uL (Normal: 150-400) - **Normal**

**LIPID PANEL**

1. Total Cholesterol: 238 mg/dL (Normal: <200) - **Abnormal (High)**
2. LDL Cholesterol: 162 mg/dL (Normal: <100) - **Abnormal (High)**
3. HDL Cholesterol: 36 mg/dL (Normal: >40) - **Abnormal (Low)**
4. Triglycerides: 188 mg/dL (Normal: <150) - **Abnormal (High)**

**METABOLIC PANEL**

1. Glucose (Fasting): 92 mg/dL (Normal: 70-99) - **Normal**
2. HbA1c: 5.3% (Normal: <5.7%) - **Normal**
3. Creatinine: 1.0 mg/dL (Normal: 0.7-1.3) - **Normal**
4. eGFR: 82 mL/min (Normal: >60) - **Normal**

**LIVER FUNCTION**

1. ALT: 28 U/L (Normal: 7-40) - **Normal**
2. AST: 25 U/L (Normal: 10-40) - **Normal**
3. Bilirubin T

In [15]:
from langchain.tools import tool
from langchain.agents import create_agent


@tool 
def diet_recommendation(condition: str)->dict:
    """Given a health condition,return a diet plan.Condition must be one of :normal,high_cholesterol,high_sugar."""
    diet_plans = {
"high_cholesterol": {
"eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
"do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
},
"high_sugar": {
"eat":        ["vegetables","wholegrains","legumes","nuts"],
"do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
    },
"normal": {
"eat":        ["vegatables","fruits","whole grains","lean protein"],
"do_not_eat": ["excessive sugar", "processed food", "trans fats"],
},
    }
    return diet_plans.get(condition,diet_plans["normal"])

In [17]:
SYSTEM_PROMPT="""
You are a helpful medical and nutrition assistant.
For the input blood work image,extract the numbers and normal range,then categorise
the condition as one of : normal,high_cholesterol,high_sugar.
Then call the appropriate tool to retrieve and present diet plan.
"""

diet_agent=create_agent(
    llm,
    tools=[diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [18]:
result = diet_agent. invoke({
"messages": [HumanMessage(content=[
{"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
{"type": "text","text": "Analyse this blood work report and suggest a diet plan."},
])]
})

print(result["messages"][-1].content)

The patient has high cholesterol and high sugar levels. 

For high cholesterol, a suitable diet plan would include eating fruits, vegetables, whole grains, and lean protein, while avoiding red meat, fried food, full-fat dairy, and processed snacks.

For high sugar levels, a suitable diet plan would include eating vegetables, whole grains, legumes, and nuts, while avoiding white rice, white sugar, junk food, and sugary drinks.
